# RAY-IMAGE N2 — Persistent Colab Experiment Notebook

This is the controlled N2 → N3 experiment runner. **Run cells top-to-bottom, one at a time.**

The key change in this notebook is persistence: checkpoints, N1 statistics, generated images, and diagnostic reports are copied to Google Drive so a Colab runtime reset does not destroy the experiment state. GitHub remains the source of truth for code; Drive stores binary experiment artifacts.

Select a T4 GPU before starting.

In [ ]:
# 1. Verify GPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. In Colab choose Runtime → Change runtime type → T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# 2. Clone the exact active project branch
%cd /content
!rm -rf anime-ai-companion
!git clone https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!git fetch origin arena/01a07cdc-anime-ai-companion
!git checkout arena/01a07cdc-anime-ai-companion
!git rev-parse --short HEAD
!pip install -q -r requirements.txt
print('Exact active branch ready.')

In [ ]:
# 3. Mount Google Drive and restore persistent experiment artifacts
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/RAY_IMAGE')
DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
DRIVE_RUNS = DRIVE_ROOT / 'runs'
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

def restore(src_name, dst):
    src = DRIVE_CKPT / src_name
    dst = Path(dst)
    if not dst.exists() and src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print('Restored:', dst, '<-', src)
    elif dst.exists():
        print('Local exists:', dst)
    else:
        print('Not found yet:', src)

restore('ray_vae_v0_2.pt', '/content/ray_vae_v0_2.pt')
restore('ray_image_v0_2_whiten.pt', '/content/ray_image_v0_2_whiten.pt')
if (DRIVE_RUNS / 'n1' / 'vae_latent_stats.json').exists():
    Path('/content/n1').mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_RUNS / 'n1' / 'vae_latent_stats.json', '/content/n1/vae_latent_stats.json')
    print('Restored: /content/n1/vae_latent_stats.json')
else:
    print('N1 stats not found in Drive yet.')
print('Persistent root:', DRIVE_ROOT)

In [ ]:
# 4. Project smoke tests
!python -m ray_image.train_smoke
!python -m ray_image.probe_smoke
!python -m ray_image.whiten_smoke

In [ ]:
# 5. Prepare the deterministic toy dataset
!rm -rf data/toy
!python tools/make_toy_dataset.py --output data/toy --samples 2048 --size 64 --seed 1337
print('Toy dataset ready.')

In [ ]:
# 6. VAE stage — train only when no persistent checkpoint exists
from pathlib import Path
import shutil
vae_path = Path('/content/ray_vae_v0_2.pt')
drive_vae = Path('/content/drive/MyDrive/RAY_IMAGE/checkpoints/ray_vae_v0_2.pt')
if vae_path.exists():
    print('VAE checkpoint already available; skipping retraining:', vae_path)
else:
    !python -m ray_image.train_vae --manifest data/toy/manifest.jsonl --steps 1200 --batch-size 32 --save /content/ray_vae_v0_2.pt --seed 0
if not drive_vae.exists():
    shutil.copy2(vae_path, drive_vae)
    print('Saved persistent VAE:', drive_vae)
else:
    print('Persistent VAE already exists:', drive_vae)

In [ ]:
# 7. N1 latent probe — compute/restore the exact stats used by N2
from pathlib import Path
import shutil
n1_dir = Path('/content/n1')
n1_dir.mkdir(parents=True, exist_ok=True)
stats_path = n1_dir / 'vae_latent_stats.json'
drive_n1 = Path('/content/drive/MyDrive/RAY_IMAGE/runs/n1/vae_latent_stats.json')
if stats_path.exists():
    print('N1 stats already available; skipping recomputation.')
else:
    !python -m ray_image.probe_vae_latents --vae-checkpoint /content/ray_vae_v0_2.pt --manifest data/toy/manifest.jsonl --outdir /content/n1 --size 64 --seed 1337 --stats-samples 512 --stats-batch 32
if not drive_n1.exists():
    drive_n1.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(stats_path, drive_n1)
    print('Saved persistent N1 stats:', drive_n1)
else:
    print('Persistent N1 stats already exists:', drive_n1)

In [ ]:
# 8. N2 — train generator FROM SCRATCH for 8000 steps with channel-wise whitening
from pathlib import Path
import shutil
n2_path = Path('/content/ray_image_v0_2_whiten.pt')
drive_n2 = Path('/content/drive/MyDrive/RAY_IMAGE/checkpoints/ray_image_v0_2_whiten.pt')
if n2_path.exists():
    print('N2 checkpoint already available; skipping retraining:', n2_path)
else:
    !python -m ray_image.train_generator --manifest data/toy/manifest.jsonl --vae /content/ray_vae_v0_2.pt --whiten-stats /content/n1/vae_latent_stats.json --steps 8000 --batch-size 16 --save /content/ray_image_v0_2_whiten.pt
if not drive_n2.exists():
    shutil.copy2(n2_path, drive_n2)
    print('Saved persistent N2 checkpoint:', drive_n2)
else:
    print('Persistent N2 checkpoint already exists:', drive_n2)

In [ ]:
# 9. Generate the fixed 12-prompt evaluation suite and persist it
from pathlib import Path
import shutil
import subprocess, sys
prompts = [
    ('red_circle', 'a red circle'), ('red_square', 'a red square'), ('red_triangle', 'a red triangle'),
    ('green_circle', 'a green circle'), ('green_square', 'a green square'), ('green_triangle', 'a green triangle'),
    ('blue_circle', 'a blue circle'), ('blue_square', 'a blue square'), ('blue_triangle', 'a blue triangle'),
    ('yellow_circle', 'a yellow circle'), ('yellow_square', 'a yellow square'), ('yellow_triangle', 'a yellow triangle'),
]
out_dir = Path('/content/ray_suite')
out_dir.mkdir(parents=True, exist_ok=True)
for name, prompt in prompts:
    out = out_dir / f'{name}.png'
    cmd = [sys.executable, '-m', 'ray_image.generate', '--checkpoint', '/content/ray_image_v0_2_whiten.pt', '--prompt', prompt, '--steps', '50', '--seed', '42', '--output', str(out)]
    subprocess.run(cmd, check=True)
print('Generated', len(list(out_dir.glob('*.png'))), 'images.')
drive_suite = Path('/content/drive/MyDrive/RAY_IMAGE/runs/N2/ray_suite')
drive_suite.mkdir(parents=True, exist_ok=True)
for p in out_dir.glob('*.png'):
    shutil.copy2(p, drive_suite / p.name)
print('Persistent image suite:', drive_suite)

In [ ]:
# 10. Evaluate N2 and persist the evaluator output
import subprocess, sys
from pathlib import Path
result = subprocess.run([sys.executable, 'tools/evaluate_toy_suite.py', '--dir', '/content/ray_suite'], capture_output=True, text=True, check=True)
print(result.stdout)
report_path = Path('/content/drive/MyDrive/RAY_IMAGE/runs/N2/evaluator_output.txt')
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(result.stdout)
print('Persistent evaluator report:', report_path)

## N3 — Text-conditioning diagnostic

N3 is **diagnostic only**. It does not train and does not modify the architecture. It inspects the trained N2 checkpoint to determine whether the color-vs-shape collapse originates in tokenization/text representation or in the way conditioning reaches the DiT.

Run this only after the N2 checkpoint exists.

In [ ]:
# 11. N3 — run the text-conditioning probe and persist the report
!rm -rf /content/n3
!python -m ray_image.probe_text_conditioning --checkpoint /content/ray_image_v0_2_whiten.pt --outdir /content/n3
from pathlib import Path
import shutil
report = Path('/content/n3/text_conditioning_report.json')
print('\n===== N3 REPORT =====')
print(report.read_text())
drive_report = Path('/content/drive/MyDrive/RAY_IMAGE/runs/N3/text_conditioning_report.json')
drive_report.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(report, drive_report)
print('\nPersistent N3 report:', drive_report)

In [ ]:
# 12. Display the generated image grid and show persistent artifact locations
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
import json
files = sorted(Path('/content/ray_suite').glob('*.png'))
thumbs = [Image.open(p).convert('RGB').resize((192, 192)) for p in files]
cols = 4
rows = (len(thumbs) + cols - 1) // cols
sheet = Image.new('RGB', (cols * 192, rows * 220), 'white')
draw = ImageDraw.Draw(sheet)
for i, (p, im) in enumerate(zip(files, thumbs)):
    x = (i % cols) * 192
    y = (i // cols) * 220
    sheet.paste(im, (x, y))
    draw.text((x + 5, y + 196), p.stem, fill='black')
display(sheet)
sheet_path = Path('/content/drive/MyDrive/RAY_IMAGE/runs/N2/N2_image_grid.png')
sheet.save(sheet_path)
print('Persistent grid:', sheet_path)
print('\nPersistent checkpoints:')
print('/content/drive/MyDrive/RAY_IMAGE/checkpoints/ray_vae_v0_2.pt')
print('/content/drive/MyDrive/RAY_IMAGE/checkpoints/ray_image_v0_2_whiten.pt')
print('/content/drive/MyDrive/RAY_IMAGE/runs/n1/vae_latent_stats.json')
print('/content/drive/MyDrive/RAY_IMAGE/runs/N2/evaluator_output.txt')
print('/content/drive/MyDrive/RAY_IMAGE/runs/N3/text_conditioning_report.json')